In [ ]:
!pip install transformers torch sentencepiece

In [ ]:
!pip install packaging ninja
!pip install flash-attn

In [ ]:
from transformers import pipeline
import torch
import pandas as pd

In [ ]:
# Load the JSON dataset
df_corpus_texts = pd.read_json('dta_corpus_texts.json', lines=True)
df_corpus_tokens = pd.read_json('dta_corpus_tokens.json', lines=True)

# Filter the rows where the year is 1913
filtered_df_texts = df_corpus_texts[df_corpus_texts['year'] == 1913][:10]
filtered_df_tokens = df_corpus_tokens[df_corpus_tokens['year'] == 1913][:10]
# Display the filtered DataFrame
print(filtered_df_texts)

# Optionally, you can save the filtered data to a new file
# filtered_df.to_csv('filtered_data.csv', index=False)

In [ ]:
#drop year column for one dataset
filtered_df_tokens_dropped = filtered_df_tokens.drop('year', axis=1)
print(filtered_df_tokens_dropped)

In [ ]:
# Merge DataFrames
combined_df = pd.concat([filtered_df_texts, filtered_df_tokens_dropped], axis=1)

# Display the merged DataFrame
print(combined_df)

In [ ]:
system_prompt = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X. 
X steht für 'andere'."""

prompt_format = "<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

In [ ]:
# create the prompts

prompt1 = """Geben Sie das STTS-Tag für die Token des Satzes zurück: {sentence} Der Satz sollte wie folgt tokenisiert werden: {tokens}. 
Geben Sie die Tags im Format TOKEN \t Tag zurück. 
Jedes Token-Tag-Paar sollte in einer neuen Zeile in der Ausgabedatei stehen, also fügen Sie nach den Tags ein Zeilenumbruchzeichen hinzu. 
Geben Sie nur das Token und das Tag aus (keine Erklärung, keine Übersetzung, kein zusätzlicher Text)."""

prompt2 = """Tokenisieren Sie die Wörter nicht weiter, sie sind bereits Tokenisiert.
Geben Sie nur den Tag aus (keine Erklärung, keine Übersetzung, kein zusätzlicher Text). Geben Sie den STTS-Tag für die Token zurück: {tokens}. """

prompt3 = """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Eingabesatz: {tokens}"""

#template 4
prompt4 = """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. Ihre Antwort soll keine Erklärungen und keine zusätzlichen Informationen enthalten. 
Eingabesatz: {tokens}"""

In [ ]:
#template 5
prompt5 = """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von POS-Tags sein, wobei jedes POS-Tag die POS-Markierung von dem Wort aus dem Eingabetext ist. 
Ihre Antwort sollte nur eine Liste von POS-Tags enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes POS-Tag die entsprechende POS-Markierung für ein Wort enthält. 
Eingabesatz: {tokens}"""

In [ ]:
#template 6->keine extra Info geschrieben als "gib nur eine Liste zurück und
#nichts anderes"
prompt6 = """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes. 
Eingabesatz: {tokens}"""

In [ ]:
#template 7 ->englische version des prompts
system_prompt7 = """You are a German linguist and expert in Part-of-Speech Tagging.
You use STTS (Stuttgart-Tubingen Tagset). 
STTS tags are ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, 
KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIDAT, PIDAT, PPER,PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVVFIN, VVIMP, VVIZU, VVPP, VVPP, VAFIN, VAIMP, VAPP, VMFIN, VMINF, VMPP, XY, $, ,$( , $. and X."""

prompt_format = "<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

prompt7 = """Please provide the POS tags for each word in the input sentence. 
The input will be a list of words in the sentence.
The output format should be a list of tuples, where each tuple consists of a word from the input text
and its corresponding POS tag label.
Your response should include only a list of tuples, in the order that the words appear in the input sentence,
with each tuple containing the corresponding POS tag label for a word."""

In [ ]:
#template 8 ->tagset mit erklärungen
system_prompt8 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
"""

#prompt_format = "<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

prompt8 = """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Eingabesatz: {tokens}"""

In [ ]:
#template 9 ->combi of template 6 und template 8
system_prompt9 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
"""

#prompt_format = "<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

prompt9 = """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes. 
Eingabesatz: {tokens}"""

In [ ]:
#Template 10: Template 9 + few shot examples
system_prompt10 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
"""

#prompt_format = "<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

prompt10 = """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes zurück. 
Hier sind ein paar Beispiele, was Sie machen sollen:
Eingabesatz: ['Balde', 'rings', 'die', 'Sterne', 'bleichen', 'Und', 'ermattet', 'von', 'Beschwerde', 'Wächsern', 'ihre', 'Wangen', 'bleichen', '.', 'Fäulnis', 'wittert', 'aus', 'der', 'Erde', '.']
Ausgabe: [('Balde','NN'),('rings','ADV'),('die','ART'),('Sterne','NN'),('bleichen','VVINF'),('Und','KON'),('ermattet','VVFIN'),('von','APPR'),('Beschwerde','NN'),('Wächsern','KOUS'),('ihre','PPOSAT'),('Wangen','NN'),('bleichen','VVINF'),('.','$.'),('Fäulnis','NN'),('wittert','VVFIN'),('aus','APPR'),('der','ART'),('Erde','NN'),('.','$.')]

Eingabesatz: ['Traurig', 'rauscht', 'das', 'Rohr', 'im', 'Tümpel', 'Und', 'sie', 'friert', 'in', 'sich', 'gekauert', '.', 'Fern', 'ein', 'Hahn', 'kräht', '.', 'Übern', 'Tümpel', 'Hart', 'und', 'grau', 'der', 'Morgen', 'schauert', '.']
Ausgabe: [('Traurig','ADJD'), ('rauscht','VVFIN'), ('das','ART'), ('Rohr','NN'), ('im','APPRART'), ('Tümpel','NN'), ('Und','KON'), ('sie','PPER'), ('friert','VVFIN'), ('in','APPR'), ('sich','PRF'), ('gekauert','VVPP'), ('.','$.'), ('Fern','NN'),('ein','ART'), ('Hahn','NN'), ('kräht','VVFIN'), ('.','$.'), ('Übern','NN'), ('Tümpel','NN'), ('Hart','NE'), ('und','KON'), ('grau','ADJD'), ('der','ART'), ('Morgen','NN'), ('schauert','VVFIN'), ('.','$.')]

Eingabesatz: ['In', 'der', 'Schmiede', 'dröhnt', 'der', 'Hammer', 'Und', 'sie', 'huscht', 'am', 'Tor', 'vorüber', '.', 'Glührot', 'schwingt', 'der', 'Knecht', 'den', 'Hammer', 'Und', 'sie', 'schaut', 'wie', 'tot', 'hinüber', '.']
Ausgabe: [('In','APPR'), ('der','ART'), ('Schmiede','NN'), ('dröhnt','VVFIN'), ('der','ART'), ('Hammer','NN'), ('Und','KON'), ('sie','PPER'), ('huscht','VVFIN'), ('am','APPART'), ('Tor','NN'), ('vorüber','PTKVZ'), ('.','$.'), ('Glührot','NN'), ('schwingt','VVFIN'), ('der','ART'), ('Knecht','NN'), ('den','ART'), ('Hammer','NN'), ('Und','KON'), ('sie','PPER'), ('schaut','VVFIN'), ('wie','KOKOM'), ('tot','ADJD'), ('hinüber','ADV'), ('.','$.')]
Eingabesatz: {tokens}"""


In [ ]:
# Function to generate POS tagging using one of the three templates
def generate_pos_tagging(sentence, tokenized_sentence, template_choice,chosen_prompt,num):
    # Select the appropriate template based on template_choice
    if template_choice == 0:
        prompt = chosen_prompt.format(sentence=sentence, tokens=tokenized_sentence)
    elif template_choice == 1:
        prompt = chosen_prompt.format(tokens=tokenized_sentence)
    else:
        raise ValueError("Invalid template choice! Choose 0 or 1")
    
    # Use the pipeline to generate POS tags
    if num == 7:
        output = generator(prompt_format.format(system_prompt=system_prompt7, prompt=prompt), do_sample=True, top_p=0.95, max_length=8192)
    elif num == 8:
        output = generator(prompt_format.format(system_prompt=system_prompt8, prompt=prompt), do_sample=True, top_p=0.95, max_length=8192)
    elif num == 9:
        output = generator(prompt_format.format(system_prompt=system_prompt9, prompt=prompt), do_sample=True, top_p=0.95, max_length=8192)
    elif num == 10:
        output = generator(prompt_format.format(system_prompt=system_prompt10, prompt=prompt), do_sample=True, top_p=0.95, max_length=8192)
    else:    
        output = generator(prompt_format.format(system_prompt=system_prompt, prompt=prompt), do_sample=True, top_p=0.95, max_length=8192)
    
    #extract only the assistant's answer
    response_text = output[0]['generated_text']
    assistant_start = response_text.find("assistant\n") + len("assistant\n")
    assistant_response = response_text[assistant_start:].strip()
    
    return assistant_response

In [ ]:
#leo-mistral-hessianai-7b-chat
generator = pipeline(model="LeoLM/leo-mistral-hessianai-7b-chat", device="cuda", torch_dtype=torch.float16) # True for flash-attn2 else False


In [ ]:
# Apply the function to each row in the dataset using template1
combined_df['generated_pos_tags_template1'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 0,prompt1,0), axis=1)

# Apply the function to each row in the dataset using template2
combined_df['generated_pos_tags_template2'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt2,0), axis=1)

# Apply the function to each row in the dataset using template3
combined_df['generated_pos_tags_template3'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt3,0), axis=1)

# Apply the function to each row in the dataset using template4
combined_df['generated_pos_tags_template4'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt4,0), axis=1)
# Apply the function to each row in the dataset using template5
combined_df['generated_pos_tags_template5'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt5,0), axis=1)
# Apply the function to each row in the dataset using template6
combined_df['generated_pos_tags_template6'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt6,0), axis=1)
# Apply the function to each row in the dataset using template7
combined_df['generated_pos_tags_template7'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt7,7), axis=1)
# Apply the function to each row in the dataset using template8
combined_df['generated_pos_tags_template8'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt8,8), axis=1)
# Apply the function to each row in the dataset using template9
combined_df['generated_pos_tags_template9'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt9,9), axis=1)
# Apply the function to each row in the dataset using template10
combined_df['generated_pos_tags_template10'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], 1,prompt10,10), axis=1)

In [ ]:
import re

In [ ]:
# function to extract POS-Tags with Regex
def extract_pos_tags(text):
    return re.findall(r"'([A-Z$]+(?:[A-Z.,(]+)?)'", text)

In [ ]:
# function to calculate Accuracy
def calculate_accuracy(row,column):
    model_tags = extract_pos_tags(row[column])
    expected_tags = row['labels']
    correct_count = sum(1 for model_tag, expected_tag in zip(model_tags, expected_tags) if model_tag == expected_tag)
    return correct_count / len(expected_tags) * 100

In [ ]:
## List of generated tag columns to compare
generated_columns = [
    'generated_pos_tags_template1',
    'generated_pos_tags_template2',
    'generated_pos_tags_template3',
    'generated_pos_tags_template4',
    'generated_pos_tags_template5',
    'generated_pos_tags_template6',
    'generated_pos_tags_template7',
    'generated_pos_tags_template8',
    'generated_pos_tags_template9',
    'generated_pos_tags_template10' 
]

# Loop over each generated column to calculate accuracy
for column in generated_columns:
    accuracy_column_name = f'accuracy_{column}'  # Name for the accuracy column
    combined_df[accuracy_column_name] = combined_df.apply(lambda row: calculate_accuracy(row, column), axis=1)  # Apply accuracy calculation

In [ ]:
# show results
print(combined_df.head())

In [ ]:
print(combined_df["accuracy_generated_pos_tags_template2"].tolist())

In [ ]:
# Calculate average accuracy for specified columns
accuracy_columns = [
    'accuracy_generated_pos_tags_template1',
    'accuracy_generated_pos_tags_template2',
    'accuracy_generated_pos_tags_template3',
    'accuracy_generated_pos_tags_template4',
    'accuracy_generated_pos_tags_template5',
    'accuracy_generated_pos_tags_template6',
    'accuracy_generated_pos_tags_template7',
    'accuracy_generated_pos_tags_template8',
    'accuracy_generated_pos_tags_template9',
    'accuracy_generated_pos_tags_template10' 
]
average_accuracy = combined_df[accuracy_columns].mean()

#Store the result in a new DataFrame
average_accuracy_df = pd.DataFrame(average_accuracy).reset_index()
average_accuracy_df.columns = ['model', 'average_accuracy']  # Renaming columns for clarity

# Display the new DataFrame
print(average_accuracy_df)